In [ ]:
import os
import json
import jsonschema
from jsonschema import validate
from typing import List, Dict, Any
from zai import ZhipuAiClient # 换成 ZhipuAiClient
from luogu_agent.core.constant import *

print("--- [TestInit] 成功加载 BIGMODEL_API_KEY。")

# --- 2. 初始化原生 ZhipuAiClient ---
try:
    client = ZhipuAiClient(
        api_key=DMX_API_KEY,
        base_url=DMX_BASE_URL,
    )
    print("ZhipuAiClient 初始化成功。")
except Exception as e:
    print(f"ZhipuAiClient 初始化失败: {e}")
    exit()

--- [TestInit] 成功加载 BIGMODEL_API_KEY。
ZhipuAiClient 初始化成功。


In [2]:
PROBLEM_ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": {
        "detailed_solution": {
            "type": "string",
            "description": "一份详细的、步骤清晰的题解"
        },
        "sample_code": {
            "type": "string",
            "description": "一份格式良好、有注释的 C++ 参考代码"
        },
        "keywords": {
            "type": "array",
            "description": "解决此问题所需的核心知识点列表, 例如 ['动态规划', '01背包']",
            "items": {"type": "string"}
        }
    },
    "required": ["detailed_solution", "sample_code", "keywords"]
}

# --- 4. 定义本地工具 (常规 Python 函数) ---
OI_WIKI_LINKS = {
    "动态规划": "https://oi-wiki.org/dp/",
    "01背包": "https://oi-wiki.org/dp/pack/",
    "KMP": "https://oi-wiki.org/string/kmp/",
    "主席树": "https://oi-wiki.org/ds/persistent-seg/",
    "可持久化线段树": "https://oi-wiki.org/ds/persistent-seg/",
    "区间MEX": "https://oi-wiki.org/misc/mex/",
    "莫队": "https://oi-wiki.org/misc/mo-algo/"
}

def get_knowledge_link(knowledge_point: str) -> str:
    """
    (本地工具) 根据算法知识点查询其在 OI-Wiki 上的学习链接。
    """
    return OI_WIKI_LINKS.get(knowledge_point, f"https://www.google.com/search?q={knowledge_point}")

# --- 5. 序列化逻辑 (不变) ---
def _serialize_problem_data(problem_data: Dict[str, Any]) -> str:
    try:
        examples_str = ""
        for i, ex in enumerate(problem_data.get("examples", [])):
            examples_str += f"  [样例输入 {i+1}]:\n  {ex.get('input', 'N/A')}\n"
            examples_str += f"  [样例输出 {i+1}]:\n  {ex.get('output', 'N/A')}\n"
        return f"""
        [题目标题]: {problem_data.get('title', 'N/A')}
        [题目描述]: {problem_data.get('description', 'N/A')}
        [输入格式]: {problem_data.get('input_format', 'N/A')}
        [输出格式]: {problem_data.get('output_format', 'N/A')}
        [样例]:
        {examples_str}
        [数据范围和提示]: {problem_data.get('notes', 'N/A')}
        """
    except Exception as e:
        print(f"[Serialize Error] 序列化题目失败: {e}")
        return str(problem_data)

def _serialize_solutions_data(solutions_data: List[Dict[str, Any]]) -> str:
    solution_prompt_segment = ""
    for i, sol in enumerate(solutions_data):
        solution_prompt_segment += f"""
        --- 原始题解 {i+1} (作者: {sol.get('author', 'N/A')}) ---
        [思路]: {sol.get('solution_text', 'N/A')}
        [参考代码 ({sol.get('code_language', 'N/A')})]: 
        ```
        {sol.get('code', 'N/A')}
        ```
        ------------------------------------------
        """
    return solution_prompt_segment

# --- 6. 写死的测试数据 (不变) ---
TEST_PROBLEM_DATA = {
    "title": "P4137 [模板]可持久化线段树 2（区间 MEX）",
    "description": "有一个长度为 n 的数组 {a₁,a₂,…,aₙ}。m 次询问，每次询问一个区间内最小没有出现过的自然数。",
    "input_format": "第一行，两个正整数 n,m。\n第 二行，n 个非负整数 a₁,a₂,…,aₙ。\n接下来 m 行，每行两个正整数 l,r，表示一次询问。",
    "output_format": "输出 m 行，每行一个数，依次 表示每个询问的答案。",
    "examples": [{"input": "5 5\n2 1 0 2 1\n3 3\n2 3\n2 4\n1 2\n3 5", "output": "1\n2\n3\n0\n3"}],
    "notes": "对于 100% 的数据：1≤n,m≤2×10⁵，1≤l≤r≤n，0≤aᵢ≤2×10⁵。"
}
TEST_SOLUTIONS_DATA = [
    {"solution_text": "这道题求区间 MEX，很明显可以用主席树...", "author": "TestAuthor1", "code": "#include <iostream> ...", "code_language": "C++"},
    {"solution_text": "也可以用莫队。用一个桶...", "author": "TestAuthor2", "code": "// Mo's algorithm ...", "code_language": "C++"}
]

In [3]:
problem_prompt = _serialize_problem_data(TEST_PROBLEM_DATA)
solution_prompt = _serialize_solutions_data(TEST_SOLUTIONS_DATA)

# 2. 准备 System Prompt 和 User Prompt
system_prompt = f"""
你是一个算法竞赛金牌教练。
请你严格按照以下 JSON Schema 格式返回你的分析报告：
{json.dumps(PROBLEM_ANALYSIS_SCHEMA, indent=2, ensure_ascii=False)}
"""

user_prompt = f"""
请*综合分析*以下题目信息和*多份*爬取来的原始题解，
生成一份*全新的*、*高质量*的分析报告。

[题目完整信息]:
{problem_prompt}

[爬取的多份原始题解参考]:
{solution_prompt}
"""

In [7]:
print(f"正在调用 {client.base_url} 上的 glm-4.6...")
response = client.chat.completions.create(
    model="glm-4.6", 
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    thinking={"type": "disabled"},
    # response_format={"type": "json_object"}, 
    stream=True
)
full_content = ""
for chunk in response:
    if not chunk.choices:
        continue
    
    delta = chunk.choices[0].delta
    
    # 处理增量内容
    if hasattr(delta, 'content') and delta.content:
        full_content += delta.content
        print(delta.content, end="", flush=True)
    
    # 检查是否完成
    if chunk.choices[0].finish_reason:
        print(f"\n\n完成原因: {chunk.choices[0].finish_reason}")
        if hasattr(chunk, 'usage') and chunk.usage:
            print(f"令牌使用: 输入 {chunk.usage.prompt_tokens}, 输出 {chunk.usage.completion_tokens}")

print(f"\n\n完整内容:\n{full_content}")


正在调用 https://www.dmxapi.cn/v1 上的 glm-4.6...
```json
{
  "detailed_solution": "本题要求查询区间内的最小未出现自然数（MEX），即 Minimum Excluded Value。对于每个查询 [l, r]，我们需要找出最小的非负整数 x，使得 x 不在 a[l..r] 中出现。\n\n考虑到数据范围 n, m ≤ 2×10⁵，需要高效的数据结构。主席树（可持久化线段树）是解决此类问题的经典工具。具体思路如下：\n\n1. **预处理**：首先记录每个数 a_i 的前驱位置（即上一个出现 a_i 的位置），记为 pre[i]。如果 a_i 是第一次出现，则 pre[i] = 0。\n2. **构建主席树**：我们以位置 i 为版本，构建一棵主席树。主席树的节点记录区间内 pre 的最大值。具体来说，第 i 棵主席树维护的是前缀 [1, i] 中所有 a_j 的 pre[j] 值。\n3. **查询 MEX**：对于查询 [l, r]，我们利用第 r 棵主席树查询。MEX 的值 x 满足：在 [l, r] 中 x 未出现，即所有 a_j = x 的 j 都满足 j < l。由于 pre[j] 记录的是 a_j 的前驱位置，因此 x 未出现在 [l, r] 中的条件是：所有 pre[j] < l。因此，我们可以在主席树中查询最小的 x，使得在 [l, r] 中所有 a_j = x 的 pre[j] < l。具体实现时，从线段树的根节点开始，递归查询左子树和右子树，找到满足条件的最小 x。\n4. **复杂度**：预处理时间为 O(n)，每次查询时间为 O(log n)，总复杂度为 O((n + m) log n)。\n\n通过主席树的可持久化特性，我们能够高效地处理历史版本的查询，从而在 O(log n) 时间内回答每个区间的 MEX 查询。",
  "sample_code": "#include <iostream>\n#include <vector>\n#include <algorithm>\nusing namespace std;\n\nconst int MAXN = 2e5 + 5;\nint n, m;\nint a[MAXN], pre[MAXN], last_pos[MAX